# Comparing MCMC and SVI for Netflix Privacy Analysis

This notebook provides a comprehensive comparison between Markov Chain Monte Carlo (MCMC) using PyMC3 and Stochastic Variational Inference (SVI) using Pyro for privacy analysis on the Netflix dataset. We'll analyze the performance, accuracy, and efficiency of both methods for inferring private information.

## 1. Introduction and Setup

### Why Compare MCMC and SVI?

- **MCMC (PyMC3)**: Provides asymptotically exact inference but can be computationally expensive
- **SVI (Pyro)**: Offers faster convergence but may sacrifice some accuracy


In [1]:
# Disable FutureWarnings for cleaner output
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Path setup to access privugger code
import os, sys
sys.path.append(os.path.join("../../../"))

# Import standard libraries
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
from numpy.linalg import norm
import time
import pandas as pd
from scipy.stats import entropy
import gzip

# Configure matplotlib
plt.rcParams["figure.autolayout"] = True
plt.style.use('ggplot')

# Import privugger
import privugger as pv

## 2. Problem Definition

We're analyzing a privacy mechanism for the Netflix dataset where user ratings are anonymized. The key question is: **How much information about a user's original ratings can be inferred from the anonymized data?**

### The Netflix Anonymization Mechanism

1. We have a list of user ratings for movies (0-4, where 0 means "no rating")
2. We apply a flipping mechanism that changes ratings based on certain conditions
3. We observe the final anonymized ratings
4. We attempt to infer the original ratings

## 3. Data and Model Definition (Shared Components)

In [2]:
# Constants
NMOVIES = 30

# Helper function for categorical distribution
def zn_uniform(p_zero, n):
    """Create a categorical distribution with p_zero probability for zero and uniform for others"""
    return [p_zero] + [(1-p_zero)/n]*n

In [3]:
# Define input specifications
raw_ratings = pv.Categorical("ratings", p=zn_uniform(16500/17000, 5), num_elements=(NMOVIES))
f1 = pv.Categorical("f1", p=zn_uniform(.99, 6), num_elements=(NMOVIES))

In [4]:
# Define the anonymization function
def netflix_anonymisation(ratings, flip):
    """Apply anonymization to user ratings"""
    NMOVIES = 30
    for i in range(0, NMOVIES):
        if ratings[i] != 0 and flip[i] != 0:
            ratings[i] = flip[i]-1
    return ratings

In [5]:
# Create dataset and program
ds = pv.Dataset(input_specs=[raw_ratings, f1])
program = pv.Program('output',
                     dataset=ds,
                     output_type=pv.List(pv.Int),
                     function=netflix_anonymisation)

In [6]:
# Load observation data
with gzip.open("netflix.csv.gz", "r") as obs_file:
    obs = list(map(int, (obs_file.readline().decode('utf-8').split(','))))[1:1+NMOVIES]
print("Observed anonymized ratings:")
print(obs)

Observed anonymized ratings:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3]


In [7]:
# Add observation constraint to the program
program.add_observation(f'output=={obs}', precision=0.1)

## 4. MCMC Inference (PyMC3)

MCMC is a family of algorithms for sampling from probability distributions. PyMC3 provides an implementation focused on Bayesian statistical modeling.

In [8]:
# Run MCMC inference using PyMC3
print("Running MCMC inference with PyMC3...")
start_time_mcmc = time.time()
mcmc_trace = pv.infer(program, cores=4, draws=10000, method='pymc3', return_model=False)
mcmc_time = time.time() - start_time_mcmc
print(f"MCMC inference completed in {mcmc_time:.2f} seconds")

Running MCMC inference with PyMC3...


Multiprocess sampling (2 chains in 2 jobs)
CategoricalGibbsMetropolis: [ratings, f1]


Output()

Sampling 2 chains for 1_000 tune and 10_000 draw iterations (2_000 + 20_000 draws total) took 24 seconds.
/home/tommy/miniconda3/envs/privugger/lib/python3.12/site-packages/arviz/stats/diagnostics.py:592: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
We recommend running at least 4 chains for robust computation of convergence diagnostics


MCMC inference completed in 25.32 seconds


In [13]:
# Examine PyMC3 trace summary statistics
mcmc_summary = az.summary(mcmc_trace, var_names=['ratings'], skipna=True)
print("MCMC Summary Statistics:")
display(mcmc_summary.head())

MCMC Summary Statistics:


/home/tommy/miniconda3/envs/privugger/lib/python3.12/site-packages/arviz/stats/diagnostics.py:592: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
/home/tommy/miniconda3/envs/privugger/lib/python3.12/site-packages/arviz/stats/diagnostics.py:592: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
/home/tommy/miniconda3/envs/privugger/lib/python3.12/site-packages/arviz/stats/diagnostics.py:592: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
/home/tommy/miniconda3/envs/privugger/lib/python3.12/site-packages/arviz/stats/diagnostics.py:592: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
ratings[0],0.000,0.000,0.0,0.0,0.000,0.000,20000.0,20000.0,NaN
ratings[1],0.000,0.000,0.0,0.0,0.000,0.000,20000.0,20000.0,NaN
ratings[2],0.000,0.000,0.0,0.0,0.000,0.000,20000.0,20000.0,NaN
ratings[3],0.000,0.000,0.0,0.0,0.000,0.000,20000.0,20000.0,NaN
ratings[4],0.001,0.065,0.0,0.0,0.001,0.001,3341.0,3334.0,1.0


### Plot MCMC Posterior Distribution

In [14]:
# Plot posterior for MCMC (selected movies)
selected_indices = [0, 5, 10, 15, 20, 25]  # Select a few movies to visualize
var_names = [f'ratings[{i}]' for i in selected_indices]

# Plotting selected ratings
az.plot_posterior(mcmc_trace, var_names=var_names, figsize=(12, 8))
plt.suptitle('MCMC Posterior Distributions for Selected Movie Ratings', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.95)

KeyError: 'var names: "[\'ratings[0]\' \'ratings[5]\' \'ratings[10]\' \'ratings[15]\' \'ratings[20]\'\\n \'ratings[25]\'] are not present" in dataset'

## 5. SVI Inference (Pyro)

Stochastic Variational Inference (SVI) is an optimization-based approach that approximates the posterior distribution. Pyro is a probabilistic programming language built on PyTorch that supports SVI.

In [15]:
# Run SVI inference using Pyro
print("Running SVI inference with Pyro...")
start_time_svi = time.time()
svi_trace = pv.infer(program, cores=4, draws=10000, method='pyro', return_model=False, target_idx=None)
svi_time = time.time() - start_time_svi
print(f"SVI inference completed in {svi_time:.2f} seconds")

Running SVI inference with Pyro...
Observations detected, running SVI with 2000 steps
Step 0/2000 - Loss: 1.1971
Step 100/2000 - Loss: 1.1971
Step 200/2000 - Loss: 11.4090
Step 300/2000 - Loss: 6.3030


/home/tommy/Documents/Thesis/privugger/docs/tutorials/pyro_svi/../../../privugger/inference/inference.py:83: UserWarning: The following parameters are not used by the 'pyro' backend: 'cores' (value: 4)
  warnings.warn(
/home/tommy/miniconda3/envs/privugger/lib/python3.12/site-packages/pyro/util.py:303: UserWarning: Found vars in model but not guide: {'ratings', 'f1'}
  warnings.warn(f"Found vars in model but not guide: {bad_sites}")


Step 400/2000 - Loss: 6.3031
Step 500/2000 - Loss: 7.5840
Step 600/2000 - Loss: 7.5840
Step 700/2000 - Loss: 11.4090
Step 800/2000 - Loss: 1.1971
Step 900/2000 - Loss: 6.3031
Step 1000/2000 - Loss: 12.6899
Step 1100/2000 - Loss: 6.3031
Step 1200/2000 - Loss: 1.1971
Step 1300/2000 - Loss: 1.1971
Step 1400/2000 - Loss: 11.4090
Step 1500/2000 - Loss: 17.7959
Step 1600/2000 - Loss: 1.1971
Step 1700/2000 - Loss: 1.1971
Step 1800/2000 - Loss: 7.5840
Step 1900/2000 - Loss: 11.4090
SVI inference completed in 4.53 seconds


In [16]:
# Examine SVI trace summary statistics
svi_summary = az.summary(svi_trace, var_names=['ratings'])
print("SVI Summary Statistics:")
display(svi_summary.head())

SVI Summary Statistics:


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
ratings[0],0.090,0.566,0.0,0.0,0.006,0.004,9377.0,9350.0,1.0
ratings[1],0.091,0.568,0.0,0.0,0.006,0.004,10029.0,9982.0,1.0
ratings[2],0.083,0.555,0.0,0.0,0.006,0.004,10106.0,10196.0,1.0
ratings[3],0.091,0.572,0.0,0.0,0.006,0.004,9555.0,9650.0,1.0
ratings[4],0.096,0.590,0.0,0.0,0.006,0.004,9981.0,10008.0,1.0


### Plot SVI Posterior Distribution

In [17]:
# Plot posterior for SVI (same selected movies)
az.plot_posterior(svi_trace, var_names=var_names, figsize=(12, 8))
plt.suptitle('SVI Posterior Distributions for Selected Movie Ratings', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.95)

KeyError: 'var names: "[\'ratings[0]\' \'ratings[5]\' \'ratings[10]\' \'ratings[15]\' \'ratings[20]\'\\n \'ratings[25]\'] are not present" in dataset'

## 6. Comparative Analysis

Let's compare the MCMC and SVI approaches across several dimensions:

### Timing Comparison

In [ ]:
# Compare inference times
timing_data = pd.DataFrame({
    'Method': ['MCMC (PyMC3)', 'SVI (Pyro)'],
    'Time (seconds)': [mcmc_time, svi_time]
})

plt.figure(figsize=(10, 6))
sns.barplot(x='Method', y='Time (seconds)', data=timing_data)
plt.title('Inference Time Comparison', fontsize=16)
plt.ylabel('Time (seconds)', fontsize=14)
plt.xlabel('Method', fontsize=14)
plt.grid(axis='y')
for i, v in enumerate(timing_data['Time (seconds)']):
    plt.text(i, v + 0.1, f"{v:.2f}s", ha='center', fontweight='bold')
plt.show()

### Posterior Distribution Comparison

In [ ]:
# Function to compare posterior distributions
def compare_posteriors(mcmc_trace, svi_trace, indices, figsize=(14, 10)):
    """Compare posterior distributions between MCMC and SVI for selected indices"""
    n_indices = len(indices)
    fig, axes = plt.subplots(n_indices, 1, figsize=figsize, sharex=True)
    
    if n_indices == 1:
        axes = [axes]  # Ensure axes is a list for consistent indexing
    
    for i, idx in enumerate(indices):
        ax = axes[i]
        var_name = f'ratings[{idx}]'
        
        # Extract posterior samples
        mcmc_samples = mcmc_trace.posterior['ratings'].values[:, :, idx].flatten()
        svi_samples = svi_trace.posterior['ratings'].values[:, :, idx].flatten()
        
        # Calculate histograms for KDE plotting
        sns.kdeplot(mcmc_samples, label=f'MCMC (PyMC3)', ax=ax, color='blue')
        sns.kdeplot(svi_samples, label=f'SVI (Pyro)', ax=ax, color='red')
        
        # Add true observation if available (for reference)
        if idx < len(obs):
            ax.axvline(x=obs[idx], color='green', linestyle='--', 
                       label=f'Observed: {obs[idx]}')
            
        ax.set_title(f'Movie {idx+1} Rating Posterior', fontsize=12)
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Set x-axis ticks for categorical variables (0-4 ratings)
        ax.set_xticks(range(5))
    
    plt.tight_layout()
    plt.suptitle('MCMC vs SVI Posterior Comparison', fontsize=16, y=1.02)
    return fig, axes

In [ ]:
# Compare posteriors for selected movies
comparison_indices = [0, 5, 10, 15, 20]
fig, axes = compare_posteriors(mcmc_trace, svi_trace, comparison_indices, figsize=(14, 12))

### Statistical Divergence Analysis

Let's quantify the differences between the MCMC and SVI posteriors using KL divergence.

In [ ]:
# Function to calculate KL divergence between two categorical distributions
def calculate_kl_divergence(mcmc_samples, svi_samples, n_bins=5):
    """Calculate KL divergence between two sets of samples"""
    # Create histograms (categorical probabilities)
    mcmc_hist, _ = np.histogram(mcmc_samples, bins=n_bins, range=(0, n_bins), density=True)
    svi_hist, _ = np.histogram(svi_samples, bins=n_bins, range=(0, n_bins), density=True)
    
    # Add small constant to avoid division by zero
    epsilon = 1e-10
    mcmc_hist = mcmc_hist + epsilon
    svi_hist = svi_hist + epsilon
    
    # Normalize
    mcmc_hist = mcmc_hist / mcmc_hist.sum()
    svi_hist = svi_hist / svi_hist.sum()
    
    # Calculate KL divergence: KL(MCMC || SVI)
    kl_div = entropy(mcmc_hist, svi_hist)
    return kl_div

In [ ]:
# Calculate KL divergence for all movies
kl_divergences = []
for i in range(NMOVIES):
    mcmc_samples = mcmc_trace.posterior['ratings'].values[:, :, i].flatten()
    svi_samples = svi_trace.posterior['ratings'].values[:, :, i].flatten()
    kl_div = calculate_kl_divergence(mcmc_samples, svi_samples)
    kl_divergences.append(kl_div)

# Plot KL divergences
plt.figure(figsize=(12, 6))
plt.bar(range(NMOVIES), kl_divergences)
plt.xlabel('Movie Index', fontsize=14)
plt.ylabel('KL Divergence (MCMC || SVI)', fontsize=14)
plt.title('KL Divergence between MCMC and SVI Posteriors', fontsize=16)
plt.grid(axis='y', alpha=0.3)
plt.xticks(range(NMOVIES))
plt.axhline(y=np.mean(kl_divergences), color='r', linestyle='--', 
            label=f'Mean: {np.mean(kl_divergences):.4f}')
plt.legend()
plt.show()

## 7. Privacy Metric Computation

Let's calculate privacy metrics to assess how much information about the original data can be inferred.

In [ ]:
# Define cosine similarity function
def cos_sim(a, b):
    """Compute cosine similarity between two vectors"""
    return np.dot(a, b)/(norm(a)*norm(b))

In [ ]:
# Function to analyze and compare privacy metrics
def compare_privacy_metrics(mcmc_trace, svi_trace, obs):
    """Compare privacy metrics between MCMC and SVI"""
    # Get shapes and reshape posterior samples
    mcmc_posterior = mcmc_trace.posterior['ratings'].values.reshape([-1, NMOVIES])
    svi_posterior = svi_trace.posterior['ratings'].values.reshape([-1, NMOVIES])
    
    # Calculate cosine similarities for both methods
    mcmc_similarities = list(map(lambda b: cos_sim(obs[:NMOVIES], b), mcmc_posterior))
    svi_similarities = list(map(lambda b: cos_sim(obs[:NMOVIES], b), svi_posterior))
    
    # Plot comparison
    plt.figure(figsize=(12, 6))
    sns.kdeplot(mcmc_similarities, label='MCMC (PyMC3)', color='blue')
    sns.kdeplot(svi_similarities, label='SVI (Pyro)', color='red')
    plt.title('Distribution of Cosine Similarities to Observed Data', fontsize=16)
    plt.xlabel('Cosine Similarity', fontsize=14)
    plt.ylabel('Density', fontsize=14)
    plt.legend(fontsize=12)
    plt.grid(alpha=0.3)
    
    # Calculate summary statistics
    return {
        'MCMC': {
            'mean': np.mean(mcmc_similarities),
            'std': np.std(mcmc_similarities),
            'min': np.min(mcmc_similarities),
            'max': np.max(mcmc_similarities),
            'q25': np.percentile(mcmc_similarities, 25),
            'q50': np.percentile(mcmc_similarities, 50),
            'q75': np.percentile(mcmc_similarities, 75),
        },
        'SVI': {
            'mean': np.mean(svi_similarities),
            'std': np.std(svi_similarities),
            'min': np.min(svi_similarities),
            'max': np.max(svi_similarities),
            'q25': np.percentile(svi_similarities, 25),
            'q50': np.percentile(svi_similarities, 50),
            'q75': np.percentile(svi_similarities, 75),
        }
    }

In [ ]:
# Compare privacy metrics
privacy_metrics = compare_privacy_metrics(mcmc_trace, svi_trace, obs)
print("Privacy Metrics Summary:")
privacy_df = pd.DataFrame({
    'Metric': ['Mean Similarity', 'Std Dev', 'Min', 'Max', 'Q25', 'Median', 'Q75'],
    'MCMC (PyMC3)': [privacy_metrics['MCMC']['mean'], privacy_metrics['MCMC']['std'],
                     privacy_metrics['MCMC']['min'], privacy_metrics['MCMC']['max'],
                     privacy_metrics['MCMC']['q25'], privacy_metrics['MCMC']['q50'],
                     privacy_metrics['MCMC']['q75']],
    'SVI (Pyro)': [privacy_metrics['SVI']['mean'], privacy_metrics['SVI']['std'],
                   privacy_metrics['SVI']['min'], privacy_metrics['SVI']['max'],
                   privacy_metrics['SVI']['q25'], privacy_metrics['SVI']['q50'],
                   privacy_metrics['SVI']['q75']],
})
display(privacy_df)

## 8. Visualization Dashboard - Overall Comparison

In [ ]:
# Create a comprehensive visualization dashboard
fig = plt.figure(figsize=(15, 12))

# Layout: 2 rows, 2 columns
gs = fig.add_gridspec(2, 2)

# 1. Timing comparison
ax1 = fig.add_subplot(gs[0, 0])
sns.barplot(x='Method', y='Time (seconds)', data=timing_data, ax=ax1)
ax1.set_title('Inference Time Comparison')
ax1.set_ylabel('Time (seconds)')
for i, v in enumerate(timing_data['Time (seconds)']):
    ax1.text(i, v + 0.1, f"{v:.2f}s", ha='center', fontweight='bold')
ax1.grid(axis='y')

# 2. Speed improvement
ax2 = fig.add_subplot(gs[0, 1])
speedup = mcmc_time / svi_time
ax2.bar(['SVI vs MCMC'], [speedup], color='orange')
ax2.set_title('SVI Speed Improvement Factor')
ax2.set_ylabel('Speedup (x times faster)')
ax2.text(0, speedup/2, f"{speedup:.2f}x\nfaster", ha='center', fontweight='bold', fontsize=14)
ax2.grid(axis='y')

# 3. Average KL divergence 
ax3 = fig.add_subplot(gs[1, 0])
mean_kl = np.mean(kl_divergences)
ax3.bar(['KL(MCMC||SVI)'], [mean_kl], color='green')
ax3.set_title('Average KL Divergence')
ax3.set_ylabel('KL Divergence')
ax3.text(0, mean_kl/2, f"{mean_kl:.4f}", ha='center', fontweight='bold', fontsize=14) 
ax3.grid(axis='y')

# 4. Privacy metrics comparison
ax4 = fig.add_subplot(gs[1, 1])
metrics_to_plot = ['mean', 'std', 'q75']
labels = ['Mean Similarity', 'Std Deviation', '75th Percentile']
x = np.arange(len(metrics_to_plot))
width = 0.35
mcmc_values = [privacy_metrics['MCMC'][m] for m in metrics_to_plot]
svi_values = [privacy_metrics['SVI'][m] for m in metrics_to_plot]
ax4.bar(x - width/2, mcmc_values, width, label='MCMC (PyMC3)', color='blue')
ax4.bar(x + width/2, svi_values, width, label='SVI (Pyro)', color='red')
ax4.set_title('Privacy Metric Comparison')
ax4.set_xticks(x)
ax4.set_xticklabels(labels)
ax4.legend()
ax4.grid(axis='y')

plt.suptitle('MCMC vs SVI Comparison Dashboard', fontsize=18)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

## 9. Posterior Movie Rating Visualizations

In [ ]:
# Create a visualization of the posterior mode for each movie rating
def get_posterior_modes(trace):
    """Extract the mode (most probable value) for each movie rating"""
    modes = []
    for i in range(NMOVIES):
        samples = trace.posterior['ratings'].values[:, :, i].flatten()
        # Count occurrences of each value
        values, counts = np.unique(samples, return_counts=True)
        # Find the most frequent value
        mode_idx = np.argmax(counts)
        modes.append(int(values[mode_idx]))
    return modes